# Week 9: Prompt Engineering Interactive Examples

This notebook contains hands-on examples of the 5 core prompting patterns:
1. Zero-shot prompting
2. Few-shot prompting
3. Chain-of-thought
4. Role-based prompting
5. Function calling

## Setup

First, let's set up the OpenAI client:

In [ ]:
import os
import json
import time
from openai import OpenAI

# Initialize client (assumes OPENAI_API_KEY is set in environment)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def call_llm(prompt, model="gpt-3.5-turbo", temperature=0, max_tokens=500):
    """Helper function to call OpenAI API with logging."""
    start_time = time.time()
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_tokens=max_tokens
        )
        
        latency = (time.time() - start_time) * 1000
        tokens = response.usage.total_tokens
        
        print(f"[Latency: {latency:.1f}ms | Tokens: {tokens}]")
        
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {str(e)}"

---

## 1. Zero-Shot Prompting

Direct instructions without examples. Best for well-defined tasks.

In [ ]:
def zero_shot_classify(text, categories):
    """
    Classify text into one of the provided categories (zero-shot).
    """
    prompt = f"""Classify the following text into exactly one of these categories: {', '.join(categories)}.

Text: {text}

Category:"""
    
    return call_llm(prompt)

# Example 1: Support ticket priority
categories = ["Urgent", "High", "Normal", "Low"]
text = "Server is down, all production services affected. Customers cannot log in."

print(f"Text: {text}")
print(f"Classification: {zero_shot_classify(text, categories)}")
print()

# Example 2: Email type
categories = ["Sales", "Support", "Billing", "General"]
text = "I'd like to upgrade my subscription to the Enterprise plan."

print(f"Text: {text}")
print(f"Classification: {zero_shot_classify(text, categories)}")

### Exercise: Try Zero-Shot

Modify the categories and text below to test zero-shot classification on your own examples:

In [ ]:
# Your own zero-shot example
my_categories = ["Category A", "Category B", "Category C"]
my_text = "Your text here"

# print(zero_shot_classify(my_text, my_categories))

---

## 2. Few-Shot Prompting

Provide examples to guide the model's response.

In [ ]:
def classify_sentiment_fewshot(text):
    """
    Classify sentiment with few-shot examples.
    """
    prompt = f"""Classify the sentiment of customer reviews as Positive, Negative, or Neutral.

Examples:
Text: "This product exceeded all my expectations!" Sentiment: Positive
Text: "Terrible quality, broke after one day." Sentiment: Negative
Text: "It's okay, nothing special." Sentiment: Neutral
Text: "Best purchase I've made this year!" Sentiment: Positive
Text: "Delivery was late but the product works fine." Sentiment: Neutral
Text: "{text}" Sentiment:"""
    
    return call_llm(prompt, temperature=0)

# Test cases
test_reviews = [
    "I absolutely love this new feature!",
    "Waste of money, very disappointed.",
    "The item arrived on Tuesday as expected.",
    "Not bad, but could be better."
]

print("Sentiment Classification with Few-Shot Learning")
print("=" * 50)
for review in test_reviews:
    sentiment = classify_sentiment_fewshot(review)
    print(f"\nText: {review}")
    print(f"Sentiment: {sentiment}")

In [ ]:
# Few-shot for Named Entity Recognition
def extract_entities_fewshot(text):
    """
    Extract named entities with few-shot examples.
    """
    prompt = f"""Extract named entities (PERSON, ORGANIZATION, LOCATION) from the text.
Output as JSON.

Examples:
Text: "Apple CEO Tim Cook announced new products in Cupertino."
Output: {{"PERSON": ["Tim Cook"], "ORGANIZATION": ["Apple"], "LOCATION": ["Cupertino"]}}

Text: "Marie Curie won the Nobel Prize while working in Paris."
Output: {{"PERSON": ["Marie Curie"], "ORGANIZATION": ["Nobel Prize"], "LOCATION": ["Paris"]}}

Text: "{text}"
Output:"""
    
    response = call_llm(prompt, temperature=0)
    try:
        return json.loads(response)
    except:
        return {"raw_response": response}

# Test entity extraction
text = "Elon Musk announced that Tesla will build a new factory in Berlin next year."
entities = extract_entities_fewshot(text)

print(f"Text: {text}")
print(f"Entities: {json.dumps(entities, indent=2)}")

---

## 3. Chain-of-Thought Prompting

Encourage step-by-step reasoning for complex problems.

In [ ]:
def solve_math_cot(problem):
    """
    Solve a math problem with chain-of-thought reasoning.
    """
    prompt = f"""Solve this math problem step by step. Show your reasoning, then provide the final answer.

Problem: {problem}

Let's think through this step by step:"""
    
    return call_llm(prompt, temperature=0, max_tokens=1000)

# Example 1: Word problem
problem1 = """A bakery sells cupcakes for $3 each and cookies for $2 each.
Sarah buys 4 cupcakes and 6 cookies. She pays with a $50 bill.
How much change should she receive?"""

print("Math Problem 1:")
print(problem1)
print("\nSolution:")
print(solve_math_cot(problem1))

In [ ]:
# Example 2: Logic puzzle
problem2 = """Three people (Alice, Bob, Carol) are wearing hats.
There are 2 red hats and 1 blue hat. Each person can see the others' hats but not their own.
Alice says: "I don't know my hat color."
Bob says: "I don't know my hat color."
Carol immediately says: "I know my hat color!"
What color is Carol's hat? Explain your reasoning."""

print("Logic Puzzle:")
print(problem2)
print("\nSolution:")
print(solve_math_cot(problem2))

### Zero-Shot CoT: Just add "Let's think step by step"

In [ ]:
def solve_with_cot_trigger(problem):
    """Zero-shot CoT with trigger phrase."""
    prompt = f"""{problem}

Let's think step by step."""
    return call_llm(prompt, temperature=0)

# Compare direct vs CoT
problem = "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 balls. How many does he have now?"

print("Without CoT:")
print(call_llm(problem, temperature=0))
print("\n" + "="*50 + "\n")
print("With CoT (Zero-shot):")
print(solve_with_cot_trigger(problem))

---

## 4. Role-Based Prompting

Assign personas to shape tone and expertise.

In [ ]:
def explain_as_role(concept, role, audience):
    """
    Explain a concept from the perspective of a specific role.
    """
    prompt = f"""You are an experienced {role}. Explain the following concept to a {audience} audience.
Use analogies and examples appropriate for your expertise.

Concept: {concept}

Explanation:"""
    
    return call_llm(prompt, temperature=0.7)

# Compare different roles explaining the same concept
concept = "How neural networks learn"

roles = [
    ("high school science teacher", "14-year-old student"),
    ("computer science professor", "undergraduate student"),
    ("AI researcher", "fellow researcher")
]

for role, audience in roles:
    print(f"\n{role.upper()} explaining to {audience.upper()}")
    print("=" * 60)
    explanation = explain_as_role(concept, role, audience)
    print(explanation[:400] + "..." if len(explanation) > 400 else explanation)
    print()

In [ ]:
# Role-based code review
def code_review_role(code, language="Python"):
    prompt = f"""You are a senior {language} developer with 15 years of experience.
Review the following code for bugs, performance issues, and best practices.

```python
{code}
```

Provide specific feedback and suggest improvements."""
    
    return call_llm(prompt, temperature=0.3, max_tokens=1000)

# Code with issues
code_sample = '''
def calculate_average(numbers):
    total = 0
    for n in numbers:
        total += n
    return total / len(numbers)
'''

print("Code Review:")
print(code_review_role(code_sample))

---

## 5. Function Calling

Structured outputs for tool integration.

In [ ]:
def extract_meeting_details(text):
    """
    Extract structured meeting information using function calling.
    """
    functions = [
        {
            "name": "schedule_meeting",
            "description": "Extract meeting details from text",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "Meeting title/topic"
                    },
                    "date": {
                        "type": "string",
                        "description": "Meeting date in YYYY-MM-DD format"
                    },
                    "time": {
                        "type": "string",
                        "description": "Meeting time in HH:MM format (24-hour)"
                    },
                    "duration_minutes": {
                        "type": "integer",
                        "description": "Meeting duration in minutes"
                    },
                    "attendees": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of attendee names or emails"
                    },
                    "location": {
                        "type": "string",
                        "description": "Meeting location or video link"
                    }
                },
                "required": ["title", "date", "time"]
            }
        }
    ]
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": text}],
        functions=functions,
        function_call={{"name": "schedule_meeting"}}
    )
    
    function_call = response.choices[0].message.function_call
    if function_call:
        return json.loads(function_call.arguments)
    return {{"error": "No function call returned"}}

# Test extraction
text = """Let's schedule the quarterly review meeting for next Tuesday, March 15th at 2:30 PM.
It should be in Conference Room B and last about 90 minutes.
Invite Sarah, Mike, and the engineering team leads."""

meeting_info = extract_meeting_details(text)
print(f"Input: {text}\n")
print("Extracted meeting details:")
print(json.dumps(meeting_info, indent=2))

In [ ]:
# Multi-function routing
def route_user_request(text):
    """Route user requests to appropriate functions."""
    functions = [
        {
            "name": "search_knowledge_base",
            "description": "Search internal documentation for information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "category": {"type": "string", "enum": ["technical", "billing", "general"]}
                },
                "required": ["query"]
            }
        },
        {
            "name": "create_support_ticket",
            "description": "Create a support ticket for an issue",
            "parameters": {
                "type": "object",
                "properties": {
                    "subject": {"type": "string"},
                    "priority": {"type": "string", "enum": ["low", "medium", "high", "urgent"]},
                    "description": {"type": "string"}
                },
                "required": ["subject", "priority", "description"]
            }
        },
        {
            "name": "escalate_to_human",
            "description": "Escalate to human support agent",
            "parameters": {
                "type": "object",
                "properties": {
                    "reason": {"type": "string"}
                },
                "required": ["reason"]
            }
        }
    ]
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": text}],
        functions=functions,
        function_call="auto"
    )
    
    message = response.choices[0].message
    if message.function_call:
        return {
            "action": message.function_call.name,
            "parameters": json.loads(message.function_call.arguments)
        }
    return {"response": message.content}

# Test routing
requests = [
    "How do I reset my password?",
    "My server keeps crashing, this is critical!",
    "I want to speak to a manager about my billing issue"
]

for req in requests:
    result = route_user_request(req)
    print(f"Request: {req}")
    print(f"Routed to: {result}\n")

---

## Performance Comparison

Let's compare the performance of different prompting patterns.

In [ ]:
import time

def benchmark_prompt(prompt, model="gpt-3.5-turbo", runs=3):
    """Benchmark a prompt's latency and consistency."""
    times = []
    responses = []
    
    for _ in range(runs):
        start = time.time()
        resp = call_llm(prompt, model=model)
        times.append((time.time() - start) * 1000)
        responses.append(resp)
    
    return {
        "avg_latency_ms": sum(times) / len(times),
        "responses": responses,
        "consistent": len(set(responses)) == 1
    }

# Test sentiment classification with different approaches
test_text = "This product is absolutely amazing!"

# Zero-shot
zero_shot_prompt = f"Classify sentiment (Positive/Negative/Neutral): {test_text}"

# Few-shot
few_shot_prompt = f"""Classify sentiment.

Examples:
Text: "Great!" -> Positive
Text: "Bad!" -> Negative
Text: "{test_text}" ->"""

print("BENCHMARK: Sentiment Classification")
print("=" * 50)

print("\n1. Zero-shot:")
result = benchmark_prompt(zero_shot_prompt, runs=3)
print(f"   Avg latency: {result['avg_latency_ms']:.1f}ms")
print(f"   Consistent: {result['consistent']}")

print("\n2. Few-shot:")
result = benchmark_prompt(few_shot_prompt, runs=3)
print(f"   Avg latency: {result['avg_latency_ms']:.1f}ms")
print(f"   Consistent: {result['consistent']}")

---

## Summary

| Pattern | Best For | Latency | Consistency |
|---------|----------|---------|-------------|
| Zero-shot | Simple, clear tasks | Low | Medium |
| Few-shot | Format consistency | Low | High |
| Chain-of-thought | Complex reasoning | Medium | High |
| Role-based | Specialized tone | Low | Medium |
| Function calling | Structured output | Low | Very High |

## Best Practices

1. **Start simple** - Try zero-shot first, add complexity only if needed
2. **Use delimiters** - Separate instructions, examples, and content clearly
3. **Specify output format** - Be explicit about expected format
4. **Test systematically** - Compare different approaches for your use case
5. **Monitor costs** - Track token usage and latency
6. **Handle errors** - Plan for API failures and unexpected outputs